# Brain Tumor Segmentation with U-Net

This notebook trains a binary U-Net on the [LGG MRI Segmentation dataset](https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation). It is an educational workflow, not a clinical tool.

The repository records an original 5-epoch CPU run with test Dice **0.2513**. Fresh runs may differ because of random initialization, package versions, and hardware.

## 1. Install and import dependencies

On Kaggle, attach the `LGG MRI Segmentation` dataset before running the notebook.

In [ ]:
import glob
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras import backend as K
from tensorflow.keras import layers, models

SEED = 42
IMAGE_SIZE = 128
BATCH_SIZE = 16
EPOCHS = 5
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2. Locate image/mask pairs

The Kaggle dataset stores each MRI slice alongside a file with the same name plus `_mask`.

In [ ]:
# Kaggle's default mount point; adjust this if you run elsewhere.
DATA_ROOT = Path('/kaggle/input/lgg-mri-segmentation/kaggle_3m')
if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Attach the LGG MRI dataset or update DATA_ROOT: {DATA_ROOT}')

mask_paths = sorted(DATA_ROOT.rglob('*_mask.tif'))
image_paths = [Path(str(mask_path).replace('_mask.tif', '.tif')) for mask_path in mask_paths]
pairs = [(image_path, mask_path) for image_path, mask_path in zip(image_paths, mask_paths) if image_path.exists()]
print(f'Found {len(pairs):,} image/mask pairs')
assert pairs, 'No image/mask pairs were found.'

## 3. Load and preprocess MRI slices

Images are converted to grayscale and resized to 128×128. Masks use nearest-neighbour interpolation so their binary labels are preserved.

In [ ]:
def load_pair(image_path, mask_path, image_size=IMAGE_SIZE):
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if image is None or mask is None:
        raise ValueError(f'Could not read {image_path} or {mask_path}')
    image = cv2.resize(image, (image_size, image_size), interpolation=cv2.INTER_AREA)
    mask = cv2.resize(mask, (image_size, image_size), interpolation=cv2.INTER_NEAREST)
    image = image.astype('float32') / 255.0
    mask = (mask > 0).astype('float32')
    return image[..., np.newaxis], mask[..., np.newaxis]

X, y = zip(*(load_pair(image_path, mask_path) for image_path, mask_path in pairs))
X = np.asarray(X, dtype='float32')
y = np.asarray(y, dtype='float32')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED)
print('Train:', X_train.shape, 'Test:', X_test.shape)

## 4. Define the U-Net

The encoder learns increasingly abstract features; skip connections pass spatial detail to the decoder for pixel-level predictions.

In [ ]:
def conv_block(inputs, filters):
    x = layers.Conv2D(filters, 3, activation='relu', padding='same')(inputs)
    x = layers.Conv2D(filters, 3, activation='relu', padding='same')(x)
    return x

def build_unet(input_shape=(IMAGE_SIZE, IMAGE_SIZE, 1)):
    inputs = layers.Input(shape=input_shape)
    c1 = conv_block(inputs, 16)
    p1 = layers.MaxPooling2D()(c1)
    c2 = conv_block(p1, 32)
    p2 = layers.MaxPooling2D()(c2)
    c3 = conv_block(p2, 64)
    p3 = layers.MaxPooling2D()(c3)
    c4 = conv_block(p3, 128)
    p4 = layers.MaxPooling2D()(c4)

    bridge = conv_block(p4, 256)

    u4 = layers.Conv2DTranspose(128, 2, strides=2, padding='same')(bridge)
    c5 = conv_block(layers.Concatenate()([u4, c4]), 128)
    u3 = layers.Conv2DTranspose(64, 2, strides=2, padding='same')(c5)
    c6 = conv_block(layers.Concatenate()([u3, c3]), 64)
    u2 = layers.Conv2DTranspose(32, 2, strides=2, padding='same')(c6)
    c7 = conv_block(layers.Concatenate()([u2, c2]), 32)
    u1 = layers.Conv2DTranspose(16, 2, strides=2, padding='same')(c7)
    c8 = conv_block(layers.Concatenate()([u1, c1]), 16)
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(c8)
    return models.Model(inputs, outputs, name='unet_lgg_mri')

def dice_coefficient(y_true, y_pred, smooth=1.0):
    y_true = K.flatten(y_true)
    y_pred = K.flatten(y_pred)
    intersection = K.sum(y_true * y_pred)
    return (2.0 * intersection + smooth) / (K.sum(y_true) + K.sum(y_pred) + smooth)

model = build_unet()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', dice_coefficient])
model.summary()

## 5. Train and plot learning curves

The recorded project run used five CPU epochs with batch size 16. Change `EPOCHS` to continue training when GPU compute is available.

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_split=0.20,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1,
)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='validation')
plt.title('Binary cross-entropy loss')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['dice_coefficient'], label='train')
plt.plot(history.history['val_dice_coefficient'], label='validation')
plt.title('Dice coefficient')
plt.legend()
plt.tight_layout()
plt.savefig('training_curves.png', dpi=160, bbox_inches='tight')
plt.show()

## 6. Evaluate and visualise predictions

Metrics use the standard 0.5 threshold. The 0.2 threshold below is only for visualising weak early predictions and must not be compared directly with the reported 0.5-threshold evaluation.

In [ ]:
probabilities = model.predict(X_test, batch_size=BATCH_SIZE)
binary_predictions = (probabilities >= 0.5).astype('float32')
test_dice = dice_coefficient(tf.convert_to_tensor(y_test), tf.convert_to_tensor(binary_predictions)).numpy()
test_accuracy = (binary_predictions == y_test).mean()
print(f'Test Dice at threshold 0.5: {test_dice:.4f}')
print(f'Test pixel accuracy at threshold 0.5: {test_accuracy:.4f}')

VISUAL_THRESHOLD = 0.2
indices = np.linspace(0, len(X_test) - 1, 5, dtype=int)
fig, axes = plt.subplots(len(indices), 3, figsize=(10, 3 * len(indices)))
for row, index in enumerate(indices):
    axes[row, 0].imshow(X_test[index].squeeze(), cmap='gray')
    axes[row, 0].set_title('MRI')
    axes[row, 1].imshow(y_test[index].squeeze(), cmap='gray')
    axes[row, 1].set_title('Ground truth')
    axes[row, 2].imshow(probabilities[index].squeeze() >= VISUAL_THRESHOLD, cmap='gray')
    axes[row, 2].set_title(f'Prediction (threshold {VISUAL_THRESHOLD})')
    for ax in axes[row]:
        ax.axis('off')
plt.tight_layout()
plt.savefig('predicted_masks.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Overlay one visualisation prediction on its source MRI.
index = indices[0]
image = X_test[index].squeeze()
overlay_mask = probabilities[index].squeeze() >= VISUAL_THRESHOLD
plt.figure(figsize=(6, 6))
plt.imshow(image, cmap='gray')
plt.imshow(np.ma.masked_where(~overlay_mask, overlay_mask), cmap='Reds', alpha=0.55)
plt.title(f'Predicted tumor overlay (threshold {VISUAL_THRESHOLD})')
plt.axis('off')
plt.savefig('tumor_overlay.png', dpi=160, bbox_inches='tight')
plt.show()